<font size=10>**TASK 3 - ??**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

**Project**: *Straining the great southern Melting Pot*

**Group 8**
- Beatriz Marques 20231605
- David Carrilho 20231693
- Duarte Fernandes 20231619
- Filipe Caçador 20231707
- Mariana Calais-Pedro 20231641

*«notebook description»*

**Question**: *??*

<font color = 'red'>**IR3 – Named Entity Recognition of Culinary and Spatial Themes**
Can we automatically extract dishes and locations from restaurant reviews using Named Entity Recognition (NER), and do the patterns of these entities reveal meaningful insights about cuisine types and dining experiences?
 
(Named Entity Recognition – focusing on dishes and locations, with optional analysis of entity transitions)

----

**IR3 – Dish Entity Networks and Cuisine Clusters**
 
Can we automatically extract dishes and related entities (e.g., locations) from restaurant reviews, and do the co-occurrence patterns of these entities form meaningful clusters that reveal cuisine types and cultural links? 

(NER + Co-occurrence Analysis + Clustering)</font>

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a> 
- [1. Imports](#1)
- [2. Data](#2)
- [3. Named Entity Rcognition](#3)
    - [3.1 Specific Data Preparation](#3_1)
    - [3.2 Model Implementation](#3_2)
    - [3.3 Model Evaluation](#3_3)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [5]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
import sys
import os
import pandas as pd
import spacy
from spacy import displacy
import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

from my_utils import *
from visualizations import *
from general_preprocessing import *

# <font color='#BFD72F' size=6>**2. Data**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

In [7]:
dataset_original = load_dataset('../data/atlanta_restaurant_slice_2023.csv')

In [8]:
dataset_original.head()

,title,categoryName,website,url,reviewsCount,stars,text
0,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,"One word amazing!! The red fish, halibut, fr..."
1,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,First time here and the food is great and the ...
2,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,I recently had the pleasure of dining at Optim...
3,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,Beautiful atmosphere and delicious food. All o...
4,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,We had a wonderful dinner at the Optimist. Our...


In [9]:
dataset_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53566 entries, 0 to 53565
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         53566 non-null  object 
 1   categoryName  53566 non-null  object 
 2   website       50600 non-null  object 
 3   url           53566 non-null  object 
 4   reviewsCount  53566 non-null  int64  
 5   stars         53566 non-null  float64
 6   text          53566 non-null  object 
dtypes: float64(1), int64(1), object(5)
memory usage: 2.9+ MB


| 🏷️ **Column Name** | 📝 **Description** |
|:-------------------|:-------------------|
|**title** | Name of the restaurant |
|**categoryName** | Labels that describe the restaurant's cuisine type |
|**website** | URL of the restaurant's webpage |
|**url** | URL of the restaurant's Google Maps page |
|**reviewsCount** | Total number of reviews for the restaurant at the time of scraping |
|**stars** | Customer rating (1 to 5) |
|**text** | Text of the review |

In [10]:
dataset = dataset_original.copy()

# <font color='#BFD72F' size=6>**3. Named Entity Recognition**</font> <a class="anchor" id="3"></a>
  
[Back to TOC](#toc)

## <font color='#BFD72F' size=6>3.1 Specific Data Preparation</font> <a class="anchor" id="3_1"></a>
  
[Back to TOC](#toc)

In [11]:
# Tokenization and POS tagging using project pipeline
dataset["pos_list"] =\
      dataset["text"].map(lambda content : main_pipeline(content,
                                                            print_output=False,
                                                            no_stopwords=False,
                                                            lowercase=False,
                                                            lemmatized=False,
                                                            no_punctuation=False,
                                                            pos_tags_list='pos_list'
                                                            ))

dataset["tokens"] =\
      dataset["text"].map(lambda content : main_pipeline(content,
                                                            print_output=False,
                                                            no_stopwords=False,
                                                            lowercase=False,
                                                            lemmatized=False,
                                                            no_punctuation=False,
                                                            tokenized_output=True
                                                            ))


In [12]:
dataset['features'] = dataset.apply(lambda row: sent2features(row['tokens'], row['pos_list']), axis=1)

In [13]:
dataset["features"].sample(1, random_state=12).iloc[0]

[{'bias': 1.0,
  'word.lower()': 'had',
  'word[-3:]': 'Had',
  'word[-2:]': 'ad',
  'word.isupper()': False,
  'word.istitle()': True,
  'word.isdigit()': False,
  'postag': 'NNP',
  'postag[:2]': 'NN',
  'BOS': True,
  '+1:word.lower()': 'my',
  '+1:word.istitle()': False,
  '+1:word.isupper()': False,
  '+1:postag': 'PRP$',
  '+1:postag[:2]': 'PR'},
 {'bias': 1.0,
  'word.lower()': 'my',
  'word[-3:]': 'my',
  'word[-2:]': 'my',
  'word.isupper()': False,
  'word.istitle()': False,
  'word.isdigit()': False,
  'postag': 'PRP$',
  'postag[:2]': 'PR',
  '-1:word.lower()': 'had',
  '-1:word.istitle()': True,
  '-1:word.isupper()': False,
  '-1:postag': 'NNP',
  '-1:postag[:2]': 'NN',
  '+1:word.lower()': 'hubby',
  '+1:word.istitle()': False,
  '+1:word.isupper()': False,
  '+1:postag': 'NN',
  '+1:postag[:2]': 'NN'},
 {'bias': 1.0,
  'word.lower()': 'hubby',
  'word[-3:]': 'bby',
  'word[-2:]': 'by',
  'word.isupper()': False,
  'word.istitle()': False,
  'word.isdigit()': False,
  'p

In [14]:
nlp = spacy.load("en_core_web_sm")

equivalence_table = {"PERSON":"-per",
                     "NORP":"-grp",
                     "FAC":"-fac",
                     "ORG":"-org",
                     "GPE":"-gpe",
                     "LOC":"-geo",
                     "DATE":"-date",
                     "TIME":"-tim",
                     "WORK_OF_ART":"-art",
                     "LAW":"-law",
                     "LANGUAGE":"-lang",
                     "EVENT":"eve",
                     "PRODUCT":"-prod",
                     "PERCENT":"-pct",
                     "MONEY":"-mon",
                     "QUANTITY":"-qty",
                     "CARDINAL":"-card",
                     "ORDINAL":"-ord",
                     "":""
                     }

In [15]:
dataset["ner_labels_custom"] = dataset.apply(
    lambda row: align_bio_to_custom_tokens(
        row["text"], 
        row["tokens"],
        nlp,
        equivalence_table
    ),
    axis=1
)

In [16]:
# Sanity check
dataset[['text','tokens','ner_labels_custom']].head()

,text,tokens,ner_labels_custom
0,"One word amazing!! The red fish, halibut, fr...","[One, word, amazing, !, !, The, red, fish, ,, ...","[B-card, O, O, O, O, O, O, O, O, O, O, O, O, O..."
1,First time here and the food is great and the ...,"[First, time, here, and, the, food, is, great,...","[B-ord, O, O, O, O, O, O, O, O, O, O, O, O, O]"
2,I recently had the pleasure of dining at Optim...,"[I, recently, had, the, pleasure, of, dining, ...","[O, O, O, O, O, O, O, O, B-per, O, B-gpe, O, B..."
3,Beautiful atmosphere and delicious food. All o...,"[Beautiful, atmosphere, and, delicious, food, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4,We had a wonderful dinner at the Optimist. Our...,"[We, had, a, wonderful, dinner, at, the, Optim...","[O, O, O, O, O, O, O, B-grp, O, O, O, O, B-car..."


In [17]:
# # Expand rows so each token becomes its own row
# dataset["ner_labels"] = dataset.apply(
#     lambda row: align_bio(nlp(row["text"]), row["tokens"]),
#     axis=1
# )

In [18]:
# dataset.head()

## <font color='#BFD72F' size=6>3.2 Model Implementation</font> <a class="anchor" id="3_2"></a>
  
[Back to TOC](#toc)

In [19]:
# Quick (q1) train-test split that we can use to test our classifier
X_train_q1, X_test_q1, y_train_q1, y_test_q1 = train_test_split(
                                                        dataset["features"], 
                                                        dataset["ner_labels_custom"],
                                                        test_size=0.2,
                                                        random_state=39
                                                        )

train_index_q1 = list(X_train_q1.index)
test_index_q1 = list(X_test_q1.index)

In [20]:
# CRF model -> conditional random field model

crf = sklearn_crfsuite.CRF(algorithm='lbfgs', 
                           c1=0.1, 
                           c2=0.1, 
                           max_iterations=100, 
                           all_possible_transitions=True,
                           verbose=True 
                           )
try:
    crf.fit(X_train_q1, y_train_q1)
except AttributeError:
    pass

loading training data to CRFsuite: 100%|██████████| 42852/42852 [00:27<00:00, 1583.00it/s]



Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 1
0....1....2....3....4....5....6....7....8....9....10
Number of features: 138414
Seconds required: 4.758

L-BFGS optimization
c1: 0.100000
c2: 0.100000
num_memories: 6
max_iterations: 100
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

Iter 1   time=23.63 loss=3414369.60 active=137623 feature_norm=1.00
Iter 2   time=24.00 loss=2166900.39 active=137375 feature_norm=14.90
Iter 3   time=11.37 loss=2020279.91 active=126286 feature_norm=14.04
Iter 4   time=271.69 loss=448782.10 active=84425 feature_norm=5.83
Iter 5   time=215.62 loss=416984.60 active=97756 feature_norm=6.06
Iter 6   time=209.34 loss=398490.74 active=108162 feature_norm=5.86
Iter 7   time=109.98 loss=368463.46 active=106419 feature_norm=5.72
Iter 8   time=70.68 loss=360368.74 active=106882 feature_norm=6.07
Iter 9   time=26.58 loss=347363.04 active=97953 feature

In [21]:
# keep tokens for inspection
X_test_tokens_q1 = dataset["tokens"].loc[test_index_q1]
y_pred_q1 = crf.predict(X_test_q1)

In [22]:
# # save model
# import joblib

# models_dir = os.path.join('..','models')
# os.makedirs(models_dir, exist_ok=True)
# joblib.dump(crf, os.path.join(models_dir, 'crf_ner_template.pkl'))

## <font color='#BFD72F' size=6>3.3 Model Evaluation</font> <a class="anchor" id="3_3"></a>
  
[Back to TOC](#toc)

In [23]:
labels = list(crf.classes_)
labels.remove("O")

round(metrics.flat_f1_score(y_test_q1, y_pred_q1,average='weighted', labels=labels), 3)

0.72

In [24]:
sorted_labels = sorted(
    labels,
    key=lambda name: (name[1:], name[0])
)

print(sklearn_crfsuite.metrics.flat_classification_report(y_test_q1, y_pred_q1, labels=sorted_labels, digits=3))

              precision    recall  f1-score   support

       B-art      0.560     0.159     0.248        88
       I-art      0.378     0.170     0.234       100
      B-card      0.818     0.851     0.834      2120
      I-card      0.815     0.707     0.757       256
      B-date      0.850     0.829     0.839      1398
      I-date      0.855     0.823     0.838      1234
       B-fac      0.556     0.364     0.440       110
       I-fac      0.527     0.350     0.421       137
       B-geo      0.594     0.427     0.497        96
       I-geo      0.333     0.214     0.261        42
       B-gpe      0.794     0.705     0.747      1001
       I-gpe      0.807     0.722     0.762       180
       B-grp      0.810     0.786     0.797       970
       I-grp      0.750     0.514     0.610        35
      B-lang      0.526     0.769     0.625        13
       B-law      0.429     0.300     0.353        20
       I-law      0.200     0.250     0.222        32
       B-mon      0.743    

In [26]:
dataset

,title,categoryName,website,url,reviewsCount,stars,text,pos_list,tokens,features,ner_labels_custom
0,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,"One word amazing!! The red fish, halibut, fr...","[CD, NN, NN, ., ., DT, JJ, NN, ,, NN, ,, VBN, ...","[One, word, amazing, !, !, The, red, fish, ,, ...","[{'bias': 1.0, 'word.lower()': 'one', 'word[-3...","[B-card, O, O, O, O, O, O, O, O, O, O, O, O, O..."
1,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,First time here and the food is great and the ...,"[JJ, NN, RB, CC, DT, NN, VBZ, JJ, CC, DT, NN, ...","[First, time, here, and, the, food, is, great,...","[{'bias': 1.0, 'word.lower()': 'first', 'word[...","[B-ord, O, O, O, O, O, O, O, O, O, O, O, O, O]"
2,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,I recently had the pleasure of dining at Optim...,"[PRP, RB, VBD, DT, NN, IN, VBG, IN, NNP, IN, N...","[I, recently, had, the, pleasure, of, dining, ...","[{'bias': 1.0, 'word.lower()': 'i', 'word[-3:]...","[O, O, O, O, O, O, O, O, B-per, O, B-gpe, O, B..."
3,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,Beautiful atmosphere and delicious food. All o...,"[NNP, RB, CC, JJ, NN, ., DT, IN, DT, NN, NN, N...","[Beautiful, atmosphere, and, delicious, food, ...","[{'bias': 1.0, 'word.lower()': 'beautiful', 'w...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4,The Optimist,Seafood restaurant,https://www.theoptimistrestaurant.com/,https://www.google.com/maps/place/The+Optimist...,3349,5.0,We had a wonderful dinner at the Optimist. Our...,"[PRP, VBD, DT, JJ, NN, IN, DT, NNP, ., PRP$, N...","[We, had, a, wonderful, dinner, at, the, Optim...","[{'bias': 1.0, 'word.lower()': 'we', 'word[-3:...","[O, O, O, O, O, O, O, B-grp, O, O, O, O, B-car..."
...,...,...,...,...,...,...,...,...,...,...,...
53561,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Friday night dinner was Chicken Francaise. Jor...,"[NNP, NN, NN, VBD, NNP, NNP, ., NNP, VBD, DT, ...","[Friday, night, dinner, was, Chicken, Francais...","[{'bias': 1.0, 'word.lower()': 'friday', 'word...","[B-tim, I-tim, O, O, B-per, I-per, O, B-per, O..."
53562,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Great dinner.... Yay Jordan!!!!!!,"[NNP, NN, NNP, NNP, ., ., ., ., ., .]","[Great, dinner, Yay, Jordan, !, !, !, !, !, !]","[{'bias': 1.0, 'word.lower()': 'great', 'word[...","[O, O, O, B-per, O, O, O, O, O, O]"
53563,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Jordan was our server and he was fantastic! Gr...,"[NNP, VBD, PRP$, NN, CC, PRP, VBD, JJ, ., JJ, ...","[Jordan, was, our, server, and, he, was, fanta...","[{'bias': 1.0, 'word.lower()': 'jordan', 'word...","[B-per, O, O, O, O, O, O, O, O, O, O, O]"
53564,L On North,New American restaurant,https://www.lonnorth.com/?utm_source=google&ut...,https://www.google.com/maps/place/L+On+North/@...,449,5.0,Jordan was an amazing server! Great delicious...,"[NNP, VBD, DT, JJ, NN, ., NNP, JJ, NN, CC, NN,...","[Jordan, was, an, amazing, server, !, Great, d...","[{'bias': 1.0, 'word.lower()': 'jordan', 'word...","[B-per, O, O, O, O, O, O, O, O, O, O, O, O, O,..."


In [ ]:
#pip install louvain

In [29]:
import pandas as pd
import networkx as nx
from collections import defaultdict

df = dataset.copy()

# ---------------------------------------------------
# 1. Decode BIO NER tags into entity spans
# ---------------------------------------------------

def extract_entities(tokens, labels):
    """
    Extract entities from BIO-tagged tokens.
    Returns a list of (entity_text, entity_type)
    """
    entities = []
    current = []
    current_type = None

    for tok, tag in zip(tokens, labels):
        if tag.startswith("B-"):
            # start of a new entity
            if current:
                entities.append((" ".join(current), current_type))
            current = [tok]
            current_type = tag[2:]  # remove B-
        elif tag.startswith("I-") and current:
            current.append(tok)
        else:
            # Outside tag: close existing entity
            if current:
                entities.append((" ".join(current), current_type))
                current = []
                current_type = None

    # flush last entity
    if current:
        entities.append((" ".join(current), current_type))

    return entities

# ---------------------------------------------------
# 2. Extract DISH + LOCATION + CUISINE entities
# ---------------------------------------------------

# Adjust these to match your real labels:
DISH_TAGS = {"food", "dish", "meal", "ord"}    # 'ord' might be 'order' food?
LOC_TAGS = {"loc", "geo", "car", "gpe"}        
CUISINE_TAGS = {"grp", "norp"}                 # grp might include ethnicities

def classify_entity(ent_type):
    et = ent_type.lower()
    if et in DISH_TAGS:
        return "DISH"
    elif et in LOC_TAGS:
        return "LOC"
    elif et in CUISINE_TAGS:
        return "CUISINE"
    else:
        return None  # ignore others

# ---------------------------------------------------
# 3. Build co-occurrence graph
# ---------------------------------------------------

G = nx.Graph()

for i, row in df.iterrows():
    tokens = row["tokens"]
    labels = row["ner_labels_custom"]

    entities = extract_entities(tokens, labels)

    # Keep only selected entity types
    filtered = [
        (text.lower(), classify_entity(tp))
        for text, tp in entities
        if classify_entity(tp) is not None
    ]

    # Add nodes
    for ent_text, ent_type in filtered:
        if ent_text not in G.nodes:
            G.add_node(ent_text, type=ent_type)

    # Add edges for co-occurrence inside the same review
    for i in range(len(filtered)):
        for j in range(i+1, len(filtered)):
            a = filtered[i][0]
            b = filtered[j][0]

            if G.has_edge(a, b):
                G[a][b]['weight'] += 1
            else:
                G.add_edge(a, b, weight=1)

# ---------------------------------------------------
# 4. Community Detection (Louvain)
# ---------------------------------------------------

try:
    import community  # python-louvain
    partition = community.best_partition(G, weight="weight")
except ImportError:
    print("Install python-louvain: pip install python-louvain")
    partition = {node: 0 for node in G.nodes()}  # fallback

# ---------------------------------------------------
# 5. Convert clusters to DataFrame
# ---------------------------------------------------

clusters = defaultdict(list)
for node, cluster_id in partition.items():
    clusters[cluster_id].append(node)

clusters_df = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in clusters.items()]))

clusters_df.head()


,31,1,2,111,4,5,6,7,8,10,...,831,832,833,0,33,3,48,20,27,30
0,first,atlanta,optimist,4th,southern india,kazhan,zuppa,georgia,los bartenders,el valle,...,moi,xx,colombian,the city of s. fulton,a caprese salad,salem,l north,aiden,pescaterian,tartufo
1,1st,italian,round island,chefs,NaN,NaN,NaN,mexican,atendieran,risotto,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,kinda,nino,plancha,english,NaN,NaN,NaN,texan,NaN,wild mushrooms,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,second,alfredo,NaN,swedish,NaN,NaN,NaN,noche,NaN,cheese,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,third,italy,NaN,aztec,NaN,NaN,NaN,tostitos,NaN,french,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
#pip install pyvis

In [36]:
#pip install sentence_transformers

In [39]:
#pip install hf_xet

In [42]:
#pip install jinja2

In [50]:
import pandas as pd
import networkx as nx
from collections import defaultdict, Counter
from sklearn.cluster import AgglomerativeClustering
from sentence_transformers import SentenceTransformer
from pyvis.network import Network
import numpy as np
import community  # python-louvain
from jinja2 import Template
import pkgutil

# -----------------------------------------------------------
# 1. BIO Decoder (uses dataset's tokens + ner_labels_custom)
# -----------------------------------------------------------

def extract_bio_entities(tokens, labels):
    entities = []
    current_tokens = []
    current_type = None

    for tok, tag in zip(tokens, labels):
        if tag.startswith("B-"):
            if current_tokens:
                entities.append((" ".join(current_tokens), current_type))
            current_tokens = [tok]
            current_type = tag[2:]
        elif tag.startswith("I-") and current_tokens:
            current_tokens.append(tok)
        else:
            if current_tokens:
                entities.append((" ".join(current_tokens), current_type))
            current_tokens = []
            current_type = None

    if current_tokens:
        entities.append((" ".join(current_tokens), current_type))

    return entities

# -----------------------------------------------------------
# 2. Entity classification (adjust based on your labels)
# -----------------------------------------------------------

DISH_TAGS = {"ord", "food", "dish", "meal"}
LOC_TAGS = {"loc", "geo", "gpe", "car"}
CUISINE_TAGS = {"grp", "norp"}

def classify_entity(ent_type):
    if ent_type is None:
        return None
    et = ent_type.lower()
    if et in DISH_TAGS:
        return "DISH"
    if et in LOC_TAGS:
        return "LOC"
    if et in CUISINE_TAGS:
        return "CUISINE"
    return None


# -----------------------------------------------------------
# 3. Embedding-based dish merging (dedup similar dishes)
# -----------------------------------------------------------

def merge_similar_dishes(dish_list, threshold=0.8):
    """
    Use embeddings + clustering to merge similar dish names.
    """
    if len(dish_list) == 0:
        return {}

    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(dish_list)

    # Agglomerative clustering based on cosine distance
    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric='cosine',
        linkage='average',
        distance_threshold=1 - threshold
    )

    labels = clustering.fit_predict(embeddings)

    merged = defaultdict(list)
    for dish, cluster_id in zip(dish_list, labels):
        merged[cluster_id].append(dish)

    # canonical name = shortest name
    mapping = {}
    for cid, group in merged.items():
        canonical = min(group, key=len)
        for g in group:
            mapping[g] = canonical
    
    return mapping


# -----------------------------------------------------------
# 4. Extract all dishes to build mapping
# -----------------------------------------------------------

all_dishes = []

for idx, row in df.iterrows():
    ents = extract_bio_entities(row["tokens"], row["ner_labels_custom"])
    for text, tp in ents:
        if classify_entity(tp) == "DISH":
            all_dishes.append(text.lower())

dish_freq = Counter(all_dishes)

# merge dishes using embeddings
dish_mapping = merge_similar_dishes(list(dish_freq.keys()))

def normalize_dish(d):
    return dish_mapping.get(d, d)


# -----------------------------------------------------------
# 5. Build Graph with rare entity filtering
# -----------------------------------------------------------

MIN_NODE_FREQ = 5  # remove rare entities
G = nx.Graph()
entity_frequency = Counter()

# First pass: collect all entity frequencies
for idx, row in df.iterrows():
    ents = extract_bio_entities(row["tokens"], row["ner_labels_custom"])
    for text, tp in ents:
        etype = classify_entity(tp)
        if etype:
            entity_frequency[text.lower()] += 1

# Graph building
for idx, row in df.iterrows():
    ents = extract_bio_entities(row["tokens"], row["ner_labels_custom"])

    filtered = []
    for text, tp in ents:
        etype = classify_entity(tp)
        if not etype:
            continue

        text = text.lower()

        # apply merging for dishes
        if etype == "DISH":
            text = normalize_dish(text)

        # filter rare entities
        if entity_frequency[text] >= MIN_NODE_FREQ:
            filtered.append((text, etype))

    # add nodes
    for ent_text, ent_type in filtered:
        if not G.has_node(ent_text):
            G.add_node(ent_text, type=ent_type)

    # add edges inside same review
    for i in range(len(filtered)):
        for j in range(i+1, len(filtered)):
            a, _ = filtered[i]
            b, _ = filtered[j]

            if G.has_edge(a, b):
                G[a][b]['weight'] += 1
            else:
                G.add_edge(a, b, weight=1)


# -----------------------------------------------------------
# 6. Louvain Clustering
# -----------------------------------------------------------

partition = community.best_partition(G, weight="weight")

clusters = defaultdict(list)
for node, cid in partition.items():
    clusters[cid].append(node)


# -----------------------------------------------------------
# 7. Automatic cluster naming
# -----------------------------------------------------------

def name_cluster(nodes):
    cuisine_keywords = {
        "thai": "Thai",
        "mexican": "Mexican",
        "korean": "Korean",
        "japanese": "Japanese",
        "italian": "Italian",
        "french": "French",
        "indian": "Indian",
        "chinese": "Chinese",
        "bbq": "BBQ",
        "seafood": "Seafood",
        "georgia": "Georgian | Southern",
        "southern": "Southern",
    }

    # Priority 1: cuisine terms
    for n in nodes:
        for key, label in cuisine_keywords.items():
            if key in n:
                return f"{label} Cuisine Cluster"

    # Priority 2: if mostly dishes
    if sum(G.nodes[n]['type'] == "DISH" for n in nodes) > len(nodes) / 2:
        return "Dish Similarity Cluster"

    # Priority 3: if location-heavy
    if sum(G.nodes[n]['type'] == "LOC" for n in nodes) > len(nodes) / 2:
        return "Location Cluster"

    return "Mixed Culinary Cluster"


named_clusters = {
    cid: {
        "name": name_cluster(nodes),
        "nodes": nodes
    }
    for cid, nodes in clusters.items()
}


# -----------------------------------------------------------
# 8. Interactive Graph Visualization (pyvis)
# -----------------------------------------------------------

net = Network(
    height="800px",
    width="100%",
    bgcolor="#ffffff",
    font_color="black"
)

net.force_atlas_2based()

for node, data in G.nodes(data=True):
    ntype = data["type"]
    
    if ntype == "DISH":
        color = "#ff7f0e"
    elif ntype == "CUISINE":
        color = "#2ca02c"
    else:
        color = "#1f77b4"

    net.add_node(node, label=node, color=color)

for u, v, w in G.edges(data=True):
    net.add_edge(u, v, value=w["weight"])

# Add cluster names as groups
for cid, info in named_clusters.items():
    for node in info["nodes"]:
        net.get_node(node)["group"] = info["name"]


# Assign groups for visualization
cluster_info = {
    cluster_id: {
        "name": f"Cluster {cluster_id}",
        "nodes": nodes
    }
    for cluster_id, nodes in clusters.items()
}
for cluster_id, info in cluster_info.items():
    for node in info["nodes"]:
        net.get_node(node)["group"] = info["name"]

# ---- Fix PyVis template loading (works on all versions) ----

data = pkgutil.get_data("pyvis", "templates/template.html")
if data is None:
    raise RuntimeError("PyVis template not found — check installation.")
Network.template = Template(data.decode("utf-8"))

# Save file
net.write_html("dish_entity_network.html")
print("✔ Network saved to dish_entity_network.html")




✔ Network saved to dish_entity_network.html


In [52]:
import webbrowser

webbrowser.open("dish_entity_network.html")


True